# Greedy Algorithms — From Zero to LeetCode

Read top to bottom. By the end you will understand:
- What greedy means and when it works
- How it's different from DP and Backtracking
- How to think greedily (what to be greedy about)
- 7 classic patterns with full working code

---
## 1. What is a Greedy Algorithm?

Greedy = **At every step, make the locally best choice — and never go back.**

```
You're hiking to the top of a hill.
Greedy strategy: at every step, always go in the direction that goes most uphill.
```

### Greedy vs Backtracking vs DP

| Approach | Tries all? | Undoes choices? | Remembers? | Speed |
|---|---|---|---|---|
| Backtracking | Yes | Yes | No | Slow (exponential) |
| DP | All subproblems | No | Yes | Medium (polynomial) |
| **Greedy** | **No — one path** | **No** | **No** | **Fast (linear/log)** |

### When Does Greedy Work?
Greedy only gives the correct answer when:
1. **Greedy Choice Property** — the locally best choice is also globally best
2. **Optimal Substructure** — optimal solution contains optimal solutions to subproblems

**The danger:** Greedy can give wrong answer if you pick the wrong thing to be greedy about!

```
Coins: [1, 5, 6, 9],  amount = 11

Greedy (pick largest first): 9 + 1 + 1 = 3 coins  (WRONG! 5+6 = 2 coins)
DP gives correct answer: 2 coins

Lesson: greedy doesn't always work. But for many problems it does.
```

---
## 2. The Greedy Thinking Process

Ask yourself these questions:

```
1. What am I optimizing? (minimize, maximize, count)
2. What is the greedy choice? (sort by what? pick largest? smallest? earliest end?)
3. Can I prove it works? (exchange argument: swapping to non-greedy never improves result)
```

**Common greedy strategies:**
- Sort by some key, then process in order
- Always pick the minimum or maximum available
- Make earliest deadline / smallest end time first
- Use what you have before asking for more

---
## 3. Classic Example — Activity Selection

**Problem:** Given activities with start and end times, select maximum number of non-overlapping activities.

```
Activities: [(1,3), (2,5), (3,9), (0,6), (5,7), (8,9)]
Answer: 4 activities  [(1,3), (3,9) NO... let me think]
```

**Greedy Insight:** Always pick the activity that **ends earliest** (frees up time for more activities).

Why? If you pick an activity that ends late, you block more future activities.
Picking the earliest-ending one is never worse than any other choice.

In [ ]:
def activity_selection(activities):
    # GREEDY: sort by end time
    activities.sort(key=lambda x: x[1])

    count = 1
    last_end = activities[0][1]

    for start, end in activities[1:]:
        if start >= last_end:    # no overlap
            count += 1
            last_end = end

    return count


activities = [(1,3), (2,5), (3,9), (0,6), (5,7), (8,9)]
print("Max activities:", activity_selection(activities))   # 4

# Show which ones selected
activities.sort(key=lambda x: x[1])
selected = [activities[0]]
for start, end in activities[1:]:
    if start >= selected[-1][1]:
        selected.append((start, end))
print("Selected:", selected)

---
## 4. Jump Game (LeetCode 55)

**Problem:** `nums[i]` = max jumps from index `i`. Can you reach the last index?

```
nums = [2, 3, 1, 1, 4]  ->  True   (0->1->4)
nums = [3, 2, 1, 0, 4]  ->  False  (always stuck at index 3)
```

**Greedy Insight:** Track the **farthest index reachable** so far. If current index > farthest, you're stuck.

In [ ]:
def canJump(nums):
    farthest = 0

    for i in range(len(nums)):
        if i > farthest:             # can't reach index i
            return False
        farthest = max(farthest, i + nums[i])  # update farthest reachable

    return True


print(canJump([2, 3, 1, 1, 4]))   # True
print(canJump([3, 2, 1, 0, 4]))   # False
print(canJump([0]))               # True

# Visualize
nums = [2, 3, 1, 1, 4]
farthest = 0
for i in range(len(nums)):
    if i > farthest: print(f"  index {i}: STUCK!"); break
    farthest = max(farthest, i + nums[i])
    print(f"  index {i}: nums[i]={nums[i]}, farthest={farthest}")

---
## 5. Jump Game II (LeetCode 45) — Minimum Jumps

**Problem:** Minimum number of jumps to reach last index.

```
nums = [2, 3, 1, 1, 4]  ->  2 jumps  (0->1->4)
```

**Greedy Insight:** At each "level" (range of reachable indices), take one jump to the farthest possible position.

In [ ]:
def jump(nums):
    jumps = 0
    current_end = 0    # end of current jump's range
    farthest = 0       # farthest we can reach

    for i in range(len(nums) - 1):   # don't need to jump from last index
        farthest = max(farthest, i + nums[i])

        if i == current_end:         # used up all positions in this jump
            jumps += 1               # must take another jump
            current_end = farthest   # land at farthest reachable

    return jumps


print(jump([2, 3, 1, 1, 4]))   # 2
print(jump([2, 3, 0, 1, 4]))   # 2

# Visualize
nums = [2, 3, 1, 1, 4]
jumps = current_end = farthest = 0
for i in range(len(nums)-1):
    farthest = max(farthest, i + nums[i])
    if i == current_end:
        jumps += 1
        current_end = farthest
        print(f"  Jump {jumps}: reach up to index {current_end}")

---
## 6. Gas Station (LeetCode 134)

**Problem:** N gas stations in a circle. `gas[i]` = gas available. `cost[i]` = gas to travel to next station. Find starting station to complete the circle, or -1.

```
gas  = [1, 2, 3, 4, 5]
cost = [3, 4, 5, 1, 2]
Answer: start at station 3
```

**Greedy Insights:**
1. If `sum(gas) < sum(cost)` -> impossible (-1)
2. If a valid start exists, it's the station after the last point where `tank` went negative

In [ ]:
def canCompleteCircuit(gas, cost):
    if sum(gas) < sum(cost):   # total gas not enough
        return -1

    tank = 0
    start = 0

    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:           # can't continue from 'start'
            start = i + 1      # try starting from next station
            tank = 0

    return start


print(canCompleteCircuit([1,2,3,4,5], [3,4,5,1,2]))   # 3
print(canCompleteCircuit([2,3,4], [3,4,3]))            # -1

---
## 7. Meeting Rooms II (LeetCode 253) — Min Rooms Needed

**Problem:** Given meeting intervals, find minimum number of meeting rooms needed.

```
meetings = [(0,30),(5,10),(15,20)]
Answer: 2 rooms
```

**Greedy Insight:** Sort by start time. Use a min-heap to track end times. If earliest ending meeting is done by the time next meeting starts, reuse that room.

In [ ]:
import heapq

def minMeetingRooms(intervals):
    intervals.sort(key=lambda x: x[0])   # sort by start time
    heap = []   # min-heap of end times (tracks when rooms free up)

    for start, end in intervals:
        if heap and heap[0] <= start:    # a room is free
            heapq.heappop(heap)          # reuse it
        heapq.heappush(heap, end)        # assign room, track end time

    return len(heap)   # rooms in use = answer


print(minMeetingRooms([(0,30),(5,10),(15,20)]))   # 2
print(minMeetingRooms([(7,10),(2,4)]))            # 1

---
## 8. Task Scheduler (LeetCode 621)

**Problem:** CPU tasks with cooldown `n`. Minimum time to finish all tasks?

```
tasks = ['A','A','A','B','B','B'],  n = 2
Order: A B _ A B _ A B  ->  8 units
```

**Greedy Insight:** Always schedule the most frequent remaining task first. This minimizes idle time.

In [ ]:
from collections import Counter

def leastInterval(tasks, n):
    freq = Counter(tasks)
    max_freq = max(freq.values())
    max_count = sum(1 for f in freq.values() if f == max_freq)

    # Formula: place most frequent task in "chunks" of size n+1
    # Total = (max_freq - 1) * (n + 1) + max_count
    # But can't be less than total tasks
    result = (max_freq - 1) * (n + 1) + max_count
    return max(result, len(tasks))


print(leastInterval(['A','A','A','B','B','B'], 2))   # 8
print(leastInterval(['A','A','A','B','B','B'], 0))   # 6
print(leastInterval(['A','A','A','A','B','B'], 2))   # 7

---
## 9. Merge Intervals (LeetCode 56)

**Problem:** Merge all overlapping intervals.

```
Input:  [[1,3],[2,6],[8,10],[15,18]]
Output: [[1,6],[8,10],[15,18]]
```

**Greedy Insight:** Sort by start time. For each interval, either merge with last (if overlapping) or add new.

In [ ]:
def merge(intervals):
    intervals.sort(key=lambda x: x[0])   # sort by start
    merged = [intervals[0]]

    for start, end in intervals[1:]:
        last_end = merged[-1][1]

        if start <= last_end:            # OVERLAP: merge
            merged[-1][1] = max(last_end, end)
        else:                            # NO overlap: new interval
            merged.append([start, end])

    return merged


print(merge([[1,3],[2,6],[8,10],[15,18]]))   # [[1,6],[8,10],[15,18]]
print(merge([[1,4],[4,5]]))                  # [[1,5]]
print(merge([[1,4],[0,4]]))                  # [[0,4]]

---
## 10. Greedy Doesn't Always Work — Know When to Use DP

**Greedy WORKS:**
- Interval scheduling (sort by end time)
- Jump Game (track farthest reachable)
- Huffman coding (min-heap)
- Dijkstra's (greedy shortest path with heap)
- Kruskal's / Prim's (min spanning tree)

**Greedy FAILS (use DP):**
- 0/1 Knapsack (can't just pick heaviest value density)
- Coin Change with arbitrary coins
- Longest Path in a graph
- Matrix Chain Multiplication

**Quick test:** Can you prove that NOT making the greedy choice always leads to a worse or equal result? If yes -> greedy works.

---
## 11. LeetCode Greedy Patterns Cheatsheet

| Pattern | What to sort/track | Example Problems |
|---|---|---|
| Interval scheduling | Sort by end time | Meeting Rooms, Activity Selection |
| Interval merging | Sort by start time | Merge Intervals, Insert Interval |
| Jump/Reach | Track max reachable | Jump Game I & II |
| Min rooms/containers | Sort + min-heap | Meeting Rooms II |
| Running sum / balance | Linear scan with counter | Gas Station, Candy |
| Task scheduling | Max frequency first | Task Scheduler |
| Two-pointer greedy | Sort + two pointers | Two Sum (sorted), Boats |

---
## Summary

```
WHAT IS GREEDY?
  At every step, pick the locally best option and never look back.
  No recursion, no storing past results — just one forward pass.

WHEN DOES IT WORK?
  When local best = global best (Greedy Choice Property)
  Interval problems, jump/reach problems, scheduling problems

WHEN DOES IT FAIL?
  When past choices affect future in complex ways -> use DP
  Knapsack, coin change with arbitrary coins

HOW TO THINK GREEDY:
  1. What am I optimizing? (min/max what?)
  2. What should I sort by? (end time, start time, value, freq?)
  3. What do I track? (farthest, tank, rooms, last_end)

COMMON TRICKS:
  Sort by end time    -> max non-overlapping intervals
  Sort by start time  -> merge overlapping intervals
  Track max reach     -> jump game problems
  Min-heap of ends    -> min rooms/containers needed
  Most frequent first -> task/CPU scheduling
```

Now open `1-leetcode-greedy.ipynb` — every problem will click!